<a href="https://colab.research.google.com/github/morozovsolncev/ontology_of_synthesis/blob/main/matrix_2_(complex).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import networkx as nx

# =============================================================================
# КОМПЛЕКСНАЯ ПРОВЕРКА: ПРЯМОУГОЛЬНЫЕ, ПРОИЗВОЛЬНЫЕ КОМПЛЕКСНЫЕ И БОЛЬШИЕ РАЗМЕРНОСТИ
# =============================================================================

print("=" * 80)
print("КОМПЛЕКСНАЯ ПРОВЕРКА: ПРЯМОУГОЛЬНЫЕ, ПРОИЗВОЛЬНЫЕ КОМПЛЕКСНЫЕ И БОЛЬШИЕ РАЗМЕРНОСТИ")
print("=" * 80)
print()

sigma = 4.0
percentile_threshold = 90
np.random.seed(42)

# -----------------------------------------------------------------------------
# ФУНКЦИИ ГЕНЕРАЦИИ
# -----------------------------------------------------------------------------
def generate_random_hermitian(dim):
    real_part = np.random.randn(dim, dim)
    real_part = (real_part + real_part.T) / 2
    imag_part = np.random.randn(dim, dim)
    imag_part = (imag_part - imag_part.T) / 2
    return real_part + 1j * imag_part

def generate_almost_hermitian(dim, epsilon):
    H = generate_random_hermitian(dim)
    A = generate_random_hermitian(dim)
    return H + 1j * epsilon * A

def generate_random_complex(dim):
    return np.random.randn(dim, dim) + 1j * np.random.randn(dim, dim)

def generate_random_rectangular(rows, cols):
    return np.random.randn(rows, cols) + 1j * np.random.randn(rows, cols)

def commutator_norm(H1, H2):
    # Коммутатор определён только для квадратных матриц одинаковой размерности
    if H1.ndim != 2 or H2.ndim != 2:
        return 1e10
    if H1.shape != H2.shape:
        return 1e10
    if H1.shape[0] != H1.shape[1] or H2.shape[0] != H2.shape[1]:
        return 1e10
    comm = H1 @ H2 - H2 @ H1
    return np.linalg.norm(comm, ord='fro')

def resonance_probability(H1, H2, sigma):
    return np.exp(-commutator_norm(H1, H2)**2 / sigma**2)

def analyze_clusters(matrices, types, type_names):
    N = len(matrices)
    prob_matrix = np.zeros((N, N))
    for i in range(N):
        for j in range(i+1, N):
            p = resonance_probability(matrices[i], matrices[j], sigma)
            prob_matrix[i, j] = p
            prob_matrix[j, i] = p

    all_probs = prob_matrix[np.triu_indices(N, k=1)]
    if len(all_probs) == 0:
        return 0, {}, 0
    threshold = np.percentile(all_probs, percentile_threshold)

    G = nx.Graph()
    G.add_nodes_from(range(N))
    for i in range(N):
        for j in range(i+1, N):
            if prob_matrix[i, j] > threshold:
                G.add_edge(i, j)

    components = list(nx.connected_components(G))
    components_sorted = sorted(components, key=len, reverse=True)

    if components_sorted:
        main = components_sorted[0]
        size_main = len(main)
        counts = {t: 0 for t in type_names}
        for v in main:
            counts[types[v]] += 1
        return size_main, counts, len(components)
    return 0, {t: 0 for t in type_names}, 0

# =============================================================================
# ЭКСПЕРИМЕНТ 1: ПРЯМОУГОЛЬНЫЕ МАТРИЦЫ
# =============================================================================
print("=" * 80)
print("ЭКСПЕРИМЕНТ 1: ПРЯМОУГОЛЬНЫЕ МАТРИЦЫ")
print("=" * 80)

# 1A: 2x3 прямоугольные + эрмитовы 2x2 и 3x3
print("\n1A: 2x3 прямоугольные + эрмитовы 2x2 и 3x3")
N_rect_23 = 200
N_herm_2 = 200
N_herm_3 = 200

matrices = []
types = []
for _ in range(N_rect_23):
    matrices.append(generate_random_rectangular(2, 3))
    types.append('rect_2x3')
for _ in range(N_herm_2):
    matrices.append(generate_random_hermitian(2))
    types.append('herm_2')
for _ in range(N_herm_3):
    matrices.append(generate_random_hermitian(3))
    types.append('herm_3')

size_main, counts, num_clusters = analyze_clusters(matrices, types, ['rect_2x3', 'herm_2', 'herm_3'])
print(f"Главный кластер: размер = {size_main}")
print(f"Состав: rect_2x3={counts.get('rect_2x3',0)}, herm_2={counts.get('herm_2',0)}, herm_3={counts.get('herm_3',0)}")
print(f"Число кластеров: {num_clusters}")
if counts.get('rect_2x3', 0) == 0:
    print("ВЕРДИКТ: прямоугольные матрицы 2x3 НЕ ВОШЛИ в главный кластер.")
else:
    print(f"ВЕРДИКТ: прямоугольные матрицы 2x3 ВОШЛИ в главный кластер ({counts.get('rect_2x3',0)} шт.)")

# 1B: 3x4 прямоугольные + эрмитовы 3x3 и 4x4
print("\n1B: 3x4 прямоугольные + эрмитовы 3x3 и 4x4")
N_rect_34 = 200
N_herm_3b = 200
N_herm_4 = 200

matrices = []
types = []
for _ in range(N_rect_34):
    matrices.append(generate_random_rectangular(3, 4))
    types.append('rect_3x4')
for _ in range(N_herm_3b):
    matrices.append(generate_random_hermitian(3))
    types.append('herm_3')
for _ in range(N_herm_4):
    matrices.append(generate_random_hermitian(4))
    types.append('herm_4')

size_main, counts, num_clusters = analyze_clusters(matrices, types, ['rect_3x4', 'herm_3', 'herm_4'])
print(f"Главный кластер: размер = {size_main}")
print(f"Состав: rect_3x4={counts.get('rect_3x4',0)}, herm_3={counts.get('herm_3',0)}, herm_4={counts.get('herm_4',0)}")
print(f"Число кластеров: {num_clusters}")
if counts.get('rect_3x4', 0) == 0:
    print("ВЕРДИКТ: прямоугольные матрицы 3x4 НЕ ВОШЛИ в главный кластер.")
else:
    print(f"ВЕРДИКТ: прямоугольные матрицы 3x4 ВОШЛИ в главный кластер ({counts.get('rect_3x4',0)} шт.)")

# =============================================================================
# ЭКСПЕРИМЕНТ 2: ПРОИЗВОЛЬНЫЕ КОМПЛЕКСНЫЕ МАТРИЦЫ
# =============================================================================
print("\n" + "=" * 80)
print("ЭКСПЕРИМЕНТ 2: ПРОИЗВОЛЬНЫЕ КОМПЛЕКСНЫЕ МАТРИЦЫ (без структуры)")
print("=" * 80)

print("\n2A: Произвольные комплексные 3x3 + эрмитовы 3x3")
N_complex = 200
N_herm_3c = 200

matrices = []
types = []
for _ in range(N_complex):
    matrices.append(generate_random_complex(3))
    types.append('complex_3')
for _ in range(N_herm_3c):
    matrices.append(generate_random_hermitian(3))
    types.append('herm_3')

size_main, counts, num_clusters = analyze_clusters(matrices, types, ['complex_3', 'herm_3'])
print(f"Главный кластер: размер = {size_main}")
print(f"Состав: complex_3={counts.get('complex_3',0)}, herm_3={counts.get('herm_3',0)}")
print(f"Число кластеров: {num_clusters}")
if counts.get('complex_3', 0) == 0:
    print("ВЕРДИКТ: произвольные комплексные матрицы 3x3 НЕ ВОШЛИ в главный кластер.")
else:
    print(f"ВЕРДИКТ: произвольные комплексные матрицы 3x3 ВОШЛИ в главный кластер ({counts.get('complex_3',0)} шт.)")

# =============================================================================
# ЭКСПЕРИМЕНТ 3: БОЛЬШИЕ РАЗМЕРНОСТИ (зависимость включения неэрмитовых от dim)
# =============================================================================
print("\n" + "=" * 80)
print("ЭКСПЕРИМЕНТ 3: БОЛЬШИЕ РАЗМЕРНОСТИ (зависимость включения неэрмитовых от dim)")
print("=" * 80)

dims = [4, 5, 6]
epsilon = 1.0
N_herm = 200
N_nonherm = 200

print("\nРезультаты для dim = 4, 5, 6 (ε=1.0, N_herm=200, N_nonherm=200):")
print("-" * 70)

for dim in dims:
    matrices = []
    types = []
    for _ in range(N_herm):
        matrices.append(generate_random_hermitian(dim))
        types.append('herm')
    for _ in range(N_nonherm):
        matrices.append(generate_almost_hermitian(dim, epsilon))
        types.append('nonherm')

    size_main, counts, num_clusters = analyze_clusters(matrices, types, ['herm', 'nonherm'])
    frac_nonherm = counts.get('nonherm', 0) / N_nonherm if N_nonherm > 0 else 0

    print(f"dim = {dim}: главный кластер размер = {size_main}, доля nonherm = {frac_nonherm:.3f} ({counts.get('nonherm',0)} из {N_nonherm})")
    if frac_nonherm > 0.8:
        verdict = "ВЫСОКАЯ когеренция (более 80% nonherm в кластере)"
    elif frac_nonherm > 0.5:
        verdict = "СРЕДНЯЯ когеренция (50-80% nonherm в кластере)"
    elif frac_nonherm > 0.1:
        verdict = "НИЗКАЯ когеренция (10-50% nonherm в кластере)"
    else:
        verdict = "ПРАКТИЧЕСКИ НЕТ когеренции (менее 10% nonherm в кластере)"
    print(f"  ВЕРДИКТ: {verdict}")

# =============================================================================
# СВОДНЫЕ ВЫВОДЫ
# =============================================================================
print("\n" + "=" * 80)
print("СВОДНЫЕ ВЫВОДЫ")
print("=" * 80)
print("""
1. Прямоугольные матрицы: ожидается, что они НЕ ВХОДЯТ в кластер с квадратными эрмитовыми,
   так как их коммутатор не определён (возвращается большая норма → вероятность ~0).
   Это подтвердит, что только квадратные матрицы одинаковой размерности могут резонировать.

2. Произвольные комплексные матрицы: ожидается, что они НЕ ВХОДЯТ в кластер с эрмитовыми,
   так как не имеют эрмитовой структуры и их коммутаторы с эрмитовыми матрицами велики.

3. Большие размерности (d=4,5,6): ожидается, что доля неэрмитовых матриц (ε=1.0) в главном кластере
   будет уменьшаться с ростом dim. Для d=2 было 100%, для d=3 — 52.5% (из эксперимента 1.3),
   для d=4 и выше возможно дальнейшее падение. Это подтвердит гипотезу о том,
   что высокоразмерные системы более избирательны в резонансе.
""")
print("=" * 80)
print("ЭКСПЕРИМЕНТ ЗАВЕРШЁН")
print("=" * 80)

КОМПЛЕКСНАЯ ПРОВЕРКА: ПРЯМОУГОЛЬНЫЕ, ПРОИЗВОЛЬНЫЕ КОМПЛЕКСНЫЕ И БОЛЬШИЕ РАЗМЕРНОСТИ

ЭКСПЕРИМЕНТ 1: ПРЯМОУГОЛЬНЫЕ МАТРИЦЫ

1A: 2x3 прямоугольные + эрмитовы 2x2 и 3x3
Главный кластер: размер = 200
Состав: rect_2x3=0, herm_2=200, herm_3=0
Число кластеров: 202
ВЕРДИКТ: прямоугольные матрицы 2x3 НЕ ВОШЛИ в главный кластер.

1B: 3x4 прямоугольные + эрмитовы 3x3 и 4x4
Главный кластер: размер = 200
Состав: rect_3x4=0, herm_3=200, herm_4=0
Число кластеров: 202
ВЕРДИКТ: прямоугольные матрицы 3x4 НЕ ВОШЛИ в главный кластер.

ЭКСПЕРИМЕНТ 2: ПРОИЗВОЛЬНЫЕ КОМПЛЕКСНЫЕ МАТРИЦЫ (без структуры)

2A: Произвольные комплексные 3x3 + эрмитовы 3x3
Главный кластер: размер = 384
Состав: complex_3=184, herm_3=200
Число кластеров: 17
ВЕРДИКТ: произвольные комплексные матрицы 3x3 ВОШЛИ в главный кластер (184 шт.)

ЭКСПЕРИМЕНТ 3: БОЛЬШИЕ РАЗМЕРНОСТИ (зависимость включения неэрмитовых от dim)

Результаты для dim = 4, 5, 6 (ε=1.0, N_herm=200, N_nonherm=200):
---------------------------------------------------------